In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
resume_df = pd.read_csv(DATA_DIR / "Resume.csv")
job_df = pd.read_csv(DATA_DIR / "job_descriptions.csv")

In [3]:
BASE_DIR = Path("../data")
PROCESSED_DIR = Path("../data/processed")

In [4]:
resume_df.shape, job_df.shape

((2484, 4), (180370, 6))

In [5]:
resume_df.head(5)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
job_df.head(5)

,Job Title,skills,Job Description,Responsibilities,Qualifications,Experience
0,Digital Marketing Specialist,"Social media platforms (e.g., Facebook, Twitte...",Social Media Managers oversee an organizations...,"Manage and grow social media accounts, create ...",M.Tech,5 to 15 Years
1,Web Developer,"HTML, CSS, JavaScript Frontend frameworks (e.g...",Frontend Web Developers design and implement u...,"Design and code user interfaces for websites, ...",BCA,2 to 12 Years
2,Operations Manager,Quality control processes and methodologies St...,Quality Control Managers establish and enforce...,Establish and enforce quality control standard...,PhD,0 to 12 Years
3,Network Engineer,Wireless network design and architecture Wi-Fi...,"Wireless Network Engineers design, implement, ...","Design, configure, and optimize wireless netwo...",PhD,4 to 11 Years
4,Event Manager,Event planning Conference logistics Budget man...,A Conference Manager coordinates and manages c...,Specialize in conference and convention planni...,MBA,1 to 12 Years


In [7]:
resume_df = resume_df[["ID", "Resume_str", "Category"]].rename(columns={
    "ID": "resume_id",
    "Resume_str": "resume",
    "Category": "category"
})

In [8]:
job_df = job_df[["Job Title", "Job Description", "Responsibilities", "Qualifications", "skills", "Experience"]].rename(columns={
    "Job Title": "title",
    "Job Description": "job",
    "Responsibilities": "responsibilities",
    "Qualifications": "qualifications",
    "skills": "skills",
    "Experience": "experience"
})

In [9]:
resume_df.isnull().sum()

resume_id    0
resume       0
category     0
dtype: int64

In [10]:
job_df.isnull().sum()

title               0
job                 0
responsibilities    0
qualifications      0
skills              0
experience          0
dtype: int64

In [11]:
resume_df.shape, job_df.shape

((2484, 3), (180370, 6))

In [12]:
duplicate_resume  = resume_df[resume_df.duplicated()]
duplicate_resume.count()

resume_id    0
resume       0
category     0
dtype: int64

In [13]:
resume_df = resume_df.drop_duplicates()

In [14]:
duplicate_job  = job_df[job_df.duplicated()]
duplicate_job.count()

title               0
job                 0
responsibilities    0
qualifications      0
skills              0
experience          0
dtype: int64

In [15]:
job_df = job_df.drop_duplicates()

In [16]:
resume_df.shape

(2484, 3)

In [17]:
job_df.shape

(180370, 6)

In [18]:
def clean_text_light(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\b\d{10,}\b", " ", text)
    text = re.sub(r"[^a-z0-9+#\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [19]:
resume_df["clean_resume"] = resume_df["resume"].apply(clean_text_light)
for col in ["title", "job", "responsibilities", "qualifications", "skills", "experience"]:
    job_df[f"clean_{col}"] = job_df[col].apply(clean_text_light)

In [20]:
resume_df.head(2)

,resume_id,resume,category,clean_resume
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administrator marketing associate hr admini...
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist us hr operations summary versati...


In [21]:
job_df.head(2)

,title,job,responsibilities,qualifications,skills,experience,clean_title,clean_job,clean_responsibilities,clean_qualifications,clean_skills,clean_experience
0,Digital Marketing Specialist,Social Media Managers oversee an organizations...,"Manage and grow social media accounts, create ...",M.Tech,"Social media platforms (e.g., Facebook, Twitte...",5 to 15 Years,digital marketing specialist,social media managers oversee an organizations...,manage and grow social media accounts create e...,m tech,social media platforms e g facebook twitter in...,5 to 15 years
1,Web Developer,Frontend Web Developers design and implement u...,"Design and code user interfaces for websites, ...",BCA,"HTML, CSS, JavaScript Frontend frameworks (e.g...",2 to 12 Years,web developer,frontend web developers design and implement u...,design and code user interfaces for websites e...,bca,html css javascript frontend frameworks e g re...,2 to 12 years


In [22]:
job_df["full_job"] = (
    job_df["clean_title"] + " " +
    job_df["clean_title"] + " " +
    job_df["clean_job"] + " " +
    job_df["clean_responsibilities"] + " " +
    job_df["clean_qualifications"] + " " +
    job_df["clean_skills"] + " " +
    job_df["clean_experience"]
)
job_df[["title", "full_job"]].head(2)

,title,full_job
0,Digital Marketing Specialist,digital marketing specialist digital marketing...
1,Web Developer,web developer web developer frontend web devel...


In [23]:
resume_df.info(), job_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   resume_id     2484 non-null   int64
 1   resume        2484 non-null   str  
 2   category      2484 non-null   str  
 3   clean_resume  2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 77.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 180370 entries, 0 to 180369
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   title                   180370 non-null  str  
 1   job                     180370 non-null  str  
 2   responsibilities        180370 non-null  str  
 3   qualifications          180370 non-null  str  
 4   skills                  180370 non-null  str  
 5   experience              180370 non-null  str  
 6   clean_title             180370 non-null  str  
 7   clean_job               180370 non-null

(None, None)

In [24]:
resume_df["resume_word_count"] = resume_df["clean_resume"].str.split().str.len()
job_df["job_word_count"] = job_df["full_job"].str.split().str.len()
print("Before filtering:")
print("Resume:", resume_df.shape)
print("Job:", job_df.shape)
resume_df = resume_df[resume_df["resume_word_count"] > 40].copy()
job_df = job_df[job_df["job_word_count"] > 20].copy()
print("\nAfter filtering:")
print("Resume:", resume_df.shape)
print("Job:", job_df.shape)

Before filtering:
Resume: (2484, 5)
Job: (180370, 14)

After filtering:
Resume: (2483, 5)
Job: (180370, 14)


In [25]:
resume_before = len(resume_df)
job_before = len(job_df)
resume_df = resume_df.drop_duplicates(subset=["clean_resume"]).reset_index(drop=True)
job_df = job_df.drop_duplicates(subset=["title", "full_job"]).reset_index(drop=True)
print("Removed duplicate resumes:", resume_before - len(resume_df))
print("Removed duplicate jobs:", job_before - len(job_df))

Removed duplicate resumes: 2
Removed duplicate jobs: 0


In [27]:
job_df["job_id"] = job_df.index
resume_df = resume_df[
    [
        "resume_id",
        "category",
        "resume",
        "clean_resume",
        "resume_word_count",
    ]
]

job_df = job_df[
    [
        "job_id",
        "title",
        "job",
        "responsibilities",
        "qualifications",
        "skills",
        "experience",
        "clean_title",
        "clean_job",
        "clean_responsibilities",
        "clean_qualifications",
        "clean_skills",
        "clean_experience",
        "full_job",
        "job_word_count",
    ]
]

print(resume_df.shape)
print(job_df.shape)

(2481, 5)
(180370, 15)


In [28]:
print("\nSample cleaned resume:\n")
print(resume_df.loc[0, "clean_resume"][:700])
print("\nSample cleaned full job:\n")
print(job_df.loc[0, "full_job"][:700])


Sample cleaned resume:

hr administrator marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server 

Sample cleaned full job:

digital marketing specialist digital marketing specialist social media managers oversee an organizations social media presence they create and schedule content engage with followers and analyze social media metrics to drive brand awareness and eng

In [29]:
resume_out_1 = BASE_DIR / "clean_resume.csv"
job_out_1 = BASE_DIR / "clean_job.csv"

resume_out_2 = PROCESSED_DIR / "clean_resume.csv"
job_out_2 = PROCESSED_DIR / "clean_job.csv"

resume_df.to_csv(resume_out_1, index=False)
job_df.to_csv(job_out_1, index=False)

resume_df.to_csv(resume_out_2, index=False)
job_df.to_csv(job_out_2, index=False)

print("Saved:")
print("-", resume_out_1)
print("-", job_out_1)
print("-", resume_out_2)
print("-", job_out_2)

Saved:
- ..\data\clean_resume.csv
- ..\data\clean_job.csv
- ..\data\processed\clean_resume.csv
- ..\data\processed\clean_job.csv
